In [2]:

import gc
import sys
import time
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
ROOT_DIR=Path(r'D:\MyJupyter\Works\FaceScore')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0,str(ROOT_DIR))
from rating_dataset import RatingDataset
from model import SparseLandauScoreField,SparseLandauScoreFieldAblation
from others import create_baseline
WEIGHTS_DIR=ROOT_DIR/'weights'
TEST_DIR=ROOT_DIR/'data'/'test_face_data'
TABLE_DIR=ROOT_DIR/'outputs'/'tables'
MAIN_OUTPUT=TABLE_DIR/'efficiency_comparison.csv'
ABLATION_OUTPUT=TABLE_DIR/'ablation_efficiency.csv'
METHODS=['ours','EUPE','TIPSV2','QWEN35','MOBILECLIP2','PECORE','IFORMER']
VARIANTS=['full_sse','without_force','without_free_energy','without_residual','without_evidence','temperature_one']
ABLATION_CONFIGS={'full_sse':{'use_force_field':True,'use_free_energy':True,'use_direct_residual':True,'use_evidence':True,'output_temperature':1.10},'without_force':{'use_force_field':False,'use_free_energy':True,'use_direct_residual':True,'use_evidence':True,'output_temperature':1.10},'without_free_energy':{'use_force_field':True,'use_free_energy':False,'use_direct_residual':True,'use_evidence':True,'output_temperature':1.10},'without_residual':{'use_force_field':True,'use_free_energy':True,'use_direct_residual':False,'use_evidence':True,'output_temperature':1.10},'without_evidence':{'use_force_field':True,'use_free_energy':True,'use_direct_residual':True,'use_evidence':False,'output_temperature':1.10},'temperature_one':{'use_force_field':True,'use_free_energy':True,'use_direct_residual':True,'use_evidence':True,'output_temperature':1.00}}
CHECKPOINT_SEED=42
SOURCE_IMAGE_SIZE=336
BATCH_SIZE=1
BENCH_IMAGE_COUNT=8
WARMUP_STEPS=10
MEASURE_STEPS=40
TIME_ROUNDS=3
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
if DEVICE.type=='cpu':
    WARMUP_STEPS=2
    MEASURE_STEPS=8
    TIME_ROUNDS=2
if not TEST_DIR.is_dir():
    raise FileNotFoundError(f'测试集目录不存在: {TEST_DIR}')
if BENCH_IMAGE_COUNT<BATCH_SIZE or BENCH_IMAGE_COUNT%BATCH_SIZE!=0:
    raise ValueError('BENCH_IMAGE_COUNT必须是BATCH_SIZE的正整数倍')
required_paths=[WEIGHTS_DIR/f'{method}_{CHECKPOINT_SEED}.pt' for method in METHODS]+[WEIGHTS_DIR/f'{variant}_{CHECKPOINT_SEED}.pt' for variant in VARIANTS[1:]]
missing_paths=[str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('缺少推理效率评估权重:\n'+'\n'.join(missing_paths))
benchmark_dataset=RatingDataset(str(TEST_DIR),mask_dir=None,augment=False,repeat=1,image_size=SOURCE_IMAGE_SIZE,soft_target=False)
if len(benchmark_dataset)<BENCH_IMAGE_COUNT:
    raise RuntimeError(f'测试照片不足{BENCH_IMAGE_COUNT}张')
sample_indices=np.linspace(0,len(benchmark_dataset)-1,BENCH_IMAGE_COUNT,dtype=int)
benchmark_images=torch.stack([benchmark_dataset[int(i)][0] for i in sample_indices],dim=0).to(DEVICE)
BENCH_BATCHES=list(benchmark_images.split(BATCH_SIZE,dim=0))
DEVICE_NAME=torch.cuda.get_device_name(DEVICE) if DEVICE.type=='cuda' else platform.processor() or 'CPU'
print(f'设备: {DEVICE_NAME} | PyTorch {torch.__version__} | AMP={USE_AMP} | 输入={SOURCE_IMAGE_SIZE}×{SOURCE_IMAGE_SIZE} | Batch={BATCH_SIZE}')
print(f'预热{WARMUP_STEPS}次 | 每轮{MEASURE_STEPS}次 × {TIME_ROUNDS}轮')
print('计时不包含图片读取、人脸裁剪和CPU到GPU的数据传输')


设备: NVIDIA GeForce RTX 5060 Ti | PyTorch 2.10.0+cu128 | AMP=True | 输入=336×336 | Batch=1
预热10次 | 每轮40次 × 3轮
计时不包含图片读取、人脸裁剪和CPU到GPU的数据传输


In [3]:

class FrozenEncoderTokenAdapter(torch.nn.Module):
    def __init__(self,encoder,input_size,mean,std):
        super().__init__()
        self.encoder=encoder
        self.embed_dim=int(encoder.num_features)
        self.global_only=True
        self.input_size=int(input_size)
        self.register_buffer('mean',mean.detach().clone())
        self.register_buffer('std',std.detach().clone())
    def train(self,mode=True):
        super().train(mode)
        self.encoder.eval()
        return self
def load_checkpoint(path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    if not isinstance(state,dict) or not state or not all(isinstance(tensor,torch.Tensor) for tensor in state.values()):
        raise RuntimeError(f'不是直接保存的模型state_dict: {path}')
    return state
def build_sse_model(ablation=False):
    tip_model=create_baseline('TIPSV2',pretrained=False,freeze_backbone=True,dropout=0.0)
    adapter=FrozenEncoderTokenAdapter(tip_model.backbone,tip_model.input_size,tip_model.mean,tip_model.std)
    model_class=SparseLandauScoreFieldAblation if ablation else SparseLandauScoreField
    extra_config=ABLATION_CONFIGS['full_sse'] if ablation else {}
    model=model_class(pretrained=False,backbone=adapter,dropout=0.10,train_last_stage=False,width=24,prior_mix=0.0,use_entropy=False,use_high_resolution=False,scales=(224,),grid_side=16,use_evidence_head=True,evidence_base=0.1,use_global_token=True,**extra_config)
    del tip_model
    return model
def parameter_statistics(model,variant=None):
    total=sum(param.numel() for param in model.parameters())
    backbone=sum(param.numel() for param in model.backbone.parameters())
    trainable=sum(param.numel() for param in model.parameters() if param.requires_grad)
    inactive=[]
    if variant=='without_force':
        inactive.append('force_head.')
    if variant=='without_free_energy':
        inactive.append('free_energy_head.')
    if variant=='without_residual':
        inactive.append('direct_residual_head.')
    inactive_parameters=sum(param.numel() for name,param in model.named_parameters() if any(name.startswith(prefix) for prefix in inactive))
    active_trainable=sum(param.numel() for name,param in model.named_parameters() if param.requires_grad and not any(name.startswith(prefix) for prefix in inactive))
    return {'total_params':int(total),'total_params_m':total/1e6,'backbone_params_m':backbone/1e6,'registered_head_params_m':(total-backbone)/1e6,'trainable_params_m':trainable/1e6,'active_inference_params_m':(total-inactive_parameters)/1e6,'active_head_params_m':(total-backbone-inactive_parameters)/1e6,'active_trainable_params_m':active_trainable/1e6}
def forward_probabilities(model,images):
    outputs=model(images)
    log_probabilities=outputs[0] if isinstance(outputs,(tuple,list)) else outputs
    return log_probabilities.float().exp()
def forward_head_probabilities(model,features):
    outputs=model.forward_from_features(features)
    log_probabilities=outputs[0] if isinstance(outputs,(tuple,list)) else outputs
    return log_probabilities.float().exp()
@torch.inference_mode()
def benchmark_operation(operation,input_batches):
    for step in range(WARMUP_STEPS):
        with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
            output=operation(input_batches[step%len(input_batches)])
    if DEVICE.type=='cuda':
        torch.cuda.synchronize()
    durations=[]
    for round_index in range(TIME_ROUNDS):
        for step in range(MEASURE_STEPS):
            input_batch=input_batches[(round_index*MEASURE_STEPS+step)%len(input_batches)]
            if DEVICE.type=='cuda':
                event_start=torch.cuda.Event(enable_timing=True)
                event_end=torch.cuda.Event(enable_timing=True)
                event_start.record()
                with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
                    output=operation(input_batch)
                event_end.record()
                event_end.synchronize()
                milliseconds=event_start.elapsed_time(event_end)
            else:
                start_time=time.perf_counter()
                with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
                    output=operation(input_batch)
                milliseconds=(time.perf_counter()-start_time)*1000.0
            durations.append(milliseconds/input_batch.shape[0])
    durations=np.asarray(durations,dtype=np.float64)
    return {'mean_ms_per_image':float(durations.mean()),'std_ms_per_image':float(durations.std(ddof=1)),'median_ms_per_image':float(np.median(durations)),'p95_ms_per_image':float(np.quantile(durations,0.95))}
@torch.inference_mode()
def benchmark_model(model,variant=None):
    model=model.to(DEVICE).eval()
    with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
        first_prediction=forward_probabilities(model,BENCH_BATCHES[0])
    if first_prediction.shape!=(BATCH_SIZE,10) or not torch.isfinite(first_prediction).all():
        raise RuntimeError('模型未返回合法的[B,10]预测概率')
    with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
        feature_batches=[model.encode_features(batch) if hasattr(model,'encode_features') else model.extract_features(batch) for batch in BENCH_BATCHES]
    full_result=benchmark_operation(lambda batch:forward_probabilities(model,batch),BENCH_BATCHES)
    head_result=benchmark_operation(lambda features:forward_head_probabilities(model,features),feature_batches)
    info=parameter_statistics(model,variant)
    info.update({'input_resolution':int(model.backbone.input_size) if hasattr(model,'encode_features') else int(model.input_size),'batch_size':BATCH_SIZE,'device':DEVICE_NAME,'precision':'AMP_FP16' if USE_AMP else 'FP32','timer':'CUDA_events' if USE_AMP else 'perf_counter','warmup_steps':WARMUP_STEPS,'timed_steps':MEASURE_STEPS*TIME_ROUNDS,'inference_ms_mean':full_result['mean_ms_per_image'],'inference_ms_std':full_result['std_ms_per_image'],'inference_ms_median':full_result['median_ms_per_image'],'inference_ms_p95':full_result['p95_ms_per_image'],'head_ms_mean':head_result['mean_ms_per_image'],'head_ms_std':head_result['std_ms_per_image'],'throughput_images_per_second':1000.0/full_result['mean_ms_per_image']})
    return info


In [4]:

main_rows=[]
for method in METHODS:
    print(f'正在评测: {method}',flush=True)
    model=build_sse_model() if method=='ours' else create_baseline(method,pretrained=False,freeze_backbone=True,dropout=0.10)
    checkpoint=WEIGHTS_DIR/f'{method}_{CHECKPOINT_SEED}.pt'
    model.load_state_dict(load_checkpoint(checkpoint),strict=True)
    metrics=benchmark_model(model)
    main_rows.append({'method':method,'checkpoint':checkpoint.name,**metrics})
    print(f'{method}: {metrics["total_params_m"]:.3f}M | E2E={metrics["inference_ms_mean"]:.3f}ms/img | Head={metrics["head_ms_mean"]:.3f}ms/img',flush=True)
    del model
    gc.collect()
    if DEVICE.type=='cuda':
        torch.cuda.empty_cache()
main_efficiency=pd.DataFrame(main_rows)
TABLE_DIR.mkdir(parents=True,exist_ok=True)
main_efficiency.to_csv(MAIN_OUTPUT,index=False,encoding='utf-8-sig',float_format='%.6f')
print(main_efficiency[['method','total_params_m','trainable_params_m','inference_ms_mean','head_ms_mean','throughput_images_per_second']].round(4).to_string(index=False))
print(f'主对比效率表已保存: {MAIN_OUTPUT}')


正在评测: ours
ours: 86.387M | E2E=14.810ms/img | Head=2.260ms/img
正在评测: EUPE
EUPE: 21.600M | E2E=18.587ms/img | Head=0.265ms/img
正在评测: TIPSV2
TIPSV2: 86.322M | E2E=11.588ms/img | Head=0.348ms/img
正在评测: QWEN35
QWEN35: 87.422M | E2E=18.641ms/img | Head=0.255ms/img
正在评测: MOBILECLIP2
MOBILECLIP2: 10.892M | E2E=31.168ms/img | Head=0.281ms/img
正在评测: PECORE
PECORE: 23.590M | E2E=18.897ms/img | Head=0.313ms/img
正在评测: IFORMER
IFORMER: 8.526M | E2E=20.367ms/img | Head=0.234ms/img
     method  total_params_m  trainable_params_m  inference_ms_mean  head_ms_mean  throughput_images_per_second
       ours         86.3869              0.0722            14.8102        2.2601                       67.5208
       EUPE         21.6000              0.0038            18.5872        0.2647                       53.8005
     TIPSV2         86.3224              0.0077            11.5877        0.3484                       86.2982
     QWEN35         87.4222              0.0077            18.6405        0.2551    

In [5]:

reference_state=load_checkpoint(WEIGHTS_DIR/f'ours_{CHECKPOINT_SEED}.pt')
model=build_sse_model(ablation=True)
model.load_state_dict(reference_state,strict=True)
model=model.to(DEVICE).eval()
all_keys=set(model.state_dict().keys())
backbone_keys={key for key in all_keys if key.startswith('backbone.')}
ablated_rows=[]
for variant in VARIANTS:
    if variant=='full_sse':
        full_row=main_efficiency.loc[main_efficiency['method']=='ours'].iloc[0]
        metrics={key:full_row[key] for key in main_efficiency.columns if key not in ('method','checkpoint')}
        ablated_rows.append({'variant':variant,'checkpoint':f'ours_{CHECKPOINT_SEED}.pt',**metrics})
        continue
    configuration=ABLATION_CONFIGS[variant]
    for name,value in configuration.items():
        setattr(model,name,value)
    checkpoint=WEIGHTS_DIR/f'{variant}_{CHECKPOINT_SEED}.pt'
    state=load_checkpoint(checkpoint)
    if set(state.keys())!=all_keys-backbone_keys:
        raise RuntimeError(f'{checkpoint.name}紧凑权重字段与模型不匹配')
    incompatible=model.load_state_dict(state,strict=False)
    if set(incompatible.missing_keys)!=backbone_keys or incompatible.unexpected_keys:
        raise RuntimeError(f'{checkpoint.name}加载结果不符合冻结Backbone紧凑权重规范')
    metrics=benchmark_model(model,variant=variant)
    ablated_rows.append({'variant':variant,'checkpoint':checkpoint.name,**metrics})
    print(f'{variant}: Active Head={metrics["active_head_params_m"]:.5f}M | E2E={metrics["inference_ms_mean"]:.3f}ms/img | Head={metrics["head_ms_mean"]:.3f}ms/img',flush=True)
    del state
ablation_efficiency=pd.DataFrame(ablated_rows)
TABLE_DIR.mkdir(parents=True,exist_ok=True)
ablation_efficiency.to_csv(ABLATION_OUTPUT,index=False,encoding='utf-8-sig',float_format='%.6f')
print(ablation_efficiency[['variant','total_params_m','active_head_params_m','inference_ms_mean','head_ms_mean']].round(4).to_string(index=False))
print(f'消融效率表已保存: {ABLATION_OUTPUT}')
del model,reference_state
gc.collect()
if DEVICE.type=='cuda':
    torch.cuda.empty_cache()


without_force: Active Head=0.07167M | E2E=12.748ms/img | Head=1.981ms/img
without_free_energy: Active Head=0.02078M | E2E=12.992ms/img | Head=2.076ms/img
without_residual: Active Head=0.07193M | E2E=13.148ms/img | Head=2.128ms/img
without_evidence: Active Head=0.07218M | E2E=13.361ms/img | Head=2.232ms/img
temperature_one: Active Head=0.07218M | E2E=13.033ms/img | Head=2.515ms/img
            variant  total_params_m  active_head_params_m  inference_ms_mean  head_ms_mean
           full_sse         86.3869                0.0722            14.8102        2.2601
      without_force         86.3869                0.0717            12.7478        1.9807
without_free_energy         86.3869                0.0208            12.9920        2.0758
   without_residual         86.3869                0.0719            13.1481        2.1280
   without_evidence         86.3869                0.0722            13.3607        2.2320
    temperature_one         86.3869                0.0722            1